# ☁️ Cloud Security Posture Checker
### AWS IAM & S3 Policy Misconfiguration Analyzer

**Author:** Kobia Williams  
**GitHub:** [Cloud-Security-Posture-Checker](https://github.com/KobiaW/Cloud-Security-Posture-Checker)  
**Stack:** Python · Gradio · AWS Policy Analysis

---

This notebook demonstrates the **Cloud Security Posture Checker** — a security tool that audits AWS IAM and S3 JSON policy files against 7 industry-standard controls.

### Showcase Test Cases
| # | Test | Expected Result |
|---|------|----------------|
| 1 | Full Disaster | Risk Score 100 · CRITICAL RISK · all 7 findings |
| 2 | Partial Risk | Risk Score ~60 · HIGH RISK · 4 findings |
| 3 | Secure Policy | Risk Score 0 · SECURE · all checks pass ✅ |

---
Run all cells top to bottom. The final cell launches a live shareable Gradio app.


## Step 1 — Install Dependencies

In [ ]:
!pip install gradio -q
print("✅ Dependencies installed")

## Step 2 — Load Security Check Engine

In [ ]:
"""
checks.py — Core security check engine.
Each check returns a Finding dict with: id, title, severity, description, affected, passed.
"""

import re
from dataclasses import dataclass, field
from typing import Any


SEVERITY_ORDER = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3, "INFO": 4}


@dataclass
class Finding:
    id: str
    title: str
    severity: str        # CRITICAL | HIGH | MEDIUM | LOW | INFO
    description: str
    affected: list       # list of affected statement/resource strings
    passed: bool
    remediation: str = ""
    reference: str = ""


def _statements(config: dict) -> list:
    return config.get("Statement", [])


# ─────────────────────────────────────────────
# CHECK FUNCTIONS
# ─────────────────────────────────────────────

def check_wildcard_action(config: dict) -> Finding:
    """Detects Action: * (full admin access)."""
    affected = []
    for stmt in _statements(config):
        actions = stmt.get("Action", [])
        if isinstance(actions, str):
            actions = [actions]
        if "*" in actions and stmt.get("Effect") == "Allow":
            resource = stmt.get("Resource", "unknown")
            affected.append(f"Action:* on Resource:{resource}")

    return Finding(
        id="IAM-001",
        title="Wildcard Action (Action: *)",
        severity="CRITICAL",
        description=(
            "One or more statements grant ALL actions (*) to a resource. "
            "This is equivalent to full admin access and violates the principle of least privilege."
        ),
        affected=affected,
        passed=len(affected) == 0,
        remediation=(
            "Replace Action: * with only the specific actions required. "
            "For example, use ['s3:GetObject'] instead of '*' for read-only S3 access."
        ),
        reference="https://docs.aws.amazon.com/IAM/latest/UserGuide/best-practices.html#grant-least-privilege",
    )


def check_wildcard_resource(config: dict) -> Finding:
    """Detects Resource: * combined with sensitive actions."""
    sensitive_prefixes = {"iam:", "sts:", "kms:", "ec2:", "lambda:", "secretsmanager:"}
    affected = []
    for stmt in _statements(config):
        resource = stmt.get("Resource", "")
        actions = stmt.get("Action", [])
        if isinstance(actions, str):
            actions = [actions]
        if resource == "*" and stmt.get("Effect") == "Allow":
            sensitive = [a for a in actions if any(a.startswith(p) for p in sensitive_prefixes) or a == "*"]
            if sensitive:
                affected.append(f"Actions {sensitive} on Resource:*")

    return Finding(
        id="IAM-002",
        title="Wildcard Resource with Sensitive Actions",
        severity="HIGH",
        description=(
            "Sensitive AWS actions (IAM, STS, KMS, EC2, Lambda, SecretsManager) are applied "
            "to Resource: *, meaning they apply to ALL resources in the account."
        ),
        affected=affected,
        passed=len(affected) == 0,
        remediation=(
            "Scope Resource to specific ARNs. Example: "
            "'arn:aws:s3:::my-bucket/*' instead of '*'."
        ),
        reference="https://docs.aws.amazon.com/IAM/latest/UserGuide/reference_policies_elements_resource.html",
    )


def check_public_principal(config: dict) -> Finding:
    """Detects Principal: * (anonymous/public access)."""
    affected = []
    for stmt in _statements(config):
        principal = stmt.get("Principal", None)
        effect = stmt.get("Effect", "")
        if principal == "*" and effect == "Allow":
            actions = stmt.get("Action", [])
            if isinstance(actions, str):
                actions = [actions]
            affected.append(f"Principal:* allowed actions: {actions}")

    return Finding(
        id="S3-001",
        title="Public Principal (Principal: *)",
        severity="CRITICAL",
        description=(
            "One or more statements allow access from ANY principal (anonymous public). "
            "This is the most common cause of S3 data breaches."
        ),
        affected=affected,
        passed=len(affected) == 0,
        remediation=(
            "Remove Principal: * from Allow statements. "
            "Use specific IAM roles/users or apply aws:SourceAccount conditions. "
            "Enable S3 Block Public Access at the account level."
        ),
        reference="https://docs.aws.amazon.com/AmazonS3/latest/userguide/access-control-block-public-access.html",
    )


def check_missing_mfa_condition(config: dict) -> Finding:
    """Checks if sensitive actions lack MFA enforcement."""
    sensitive_prefixes = {"iam:", "sts:AssumeRole", "*"}
    affected = []
    for stmt in _statements(config):
        actions = stmt.get("Action", [])
        if isinstance(actions, str):
            actions = [actions]
        is_sensitive = any(
            a == "*" or a.startswith("iam:") or a == "sts:AssumeRole"
            for a in actions
        )
        if is_sensitive and stmt.get("Effect") == "Allow":
            conditions = stmt.get("Condition", {})
            mfa_present = (
                conditions.get("Bool", {}).get("aws:MultiFactorAuthPresent") == "true"
                or conditions.get("BoolIfExists", {}).get("aws:MultiFactorAuthPresent") == "true"
            )
            if not mfa_present:
                affected.append(f"Actions {actions} — no MFA condition")

    return Finding(
        id="IAM-003",
        title="Sensitive Actions Missing MFA Condition",
        severity="HIGH",
        description=(
            "Sensitive IAM or STS actions are permitted without requiring MFA. "
            "Without MFA enforcement, a compromised access key grants full control."
        ),
        affected=affected,
        passed=len(affected) == 0,
        remediation=(
            "Add a Condition block: "
            '{"Bool": {"aws:MultiFactorAuthPresent": "true"}} '
            "to all statements involving IAM, STS, or admin-level actions."
        ),
        reference="https://docs.aws.amazon.com/IAM/latest/UserGuide/id_credentials_mfa_configure-api-require.html",
    )


def check_allow_s3_write_public(config: dict) -> Finding:
    """Detects public write access to S3."""
    write_actions = {"s3:PutObject", "s3:DeleteObject", "s3:PutBucketPolicy", "*"}
    affected = []
    for stmt in _statements(config):
        principal = stmt.get("Principal", "")
        effect = stmt.get("Effect", "")
        actions = stmt.get("Action", [])
        if isinstance(actions, str):
            actions = [actions]
        if principal == "*" and effect == "Allow":
            dangerous = [a for a in actions if a in write_actions]
            if dangerous:
                resource = stmt.get("Resource", "unknown")
                affected.append(f"Public write actions {dangerous} on {resource}")

    return Finding(
        id="S3-002",
        title="Public S3 Write Access",
        severity="CRITICAL",
        description=(
            "S3 write or delete actions are allowed for any public principal. "
            "This allows anyone on the internet to upload, overwrite, or delete objects."
        ),
        affected=affected,
        passed=len(affected) == 0,
        remediation=(
            "Remove PutObject/DeleteObject from public-facing policies. "
            "If uploads are needed, use pre-signed URLs with expiry instead of bucket policies."
        ),
        reference="https://docs.aws.amazon.com/AmazonS3/latest/userguide/PresignedUrlUploadObject.html",
    )


def check_missing_condition_block(config: dict) -> Finding:
    """Flags Allow statements with no Condition constraints."""
    affected = []
    for i, stmt in enumerate(_statements(config)):
        if stmt.get("Effect") == "Allow" and not stmt.get("Condition"):
            actions = stmt.get("Action", [])
            if isinstance(actions, str):
                actions = [actions]
            affected.append(f"Statement {i+1} — Actions: {actions} — no Condition block")

    return Finding(
        id="IAM-004",
        title="Allow Statements Without Conditions",
        severity="MEDIUM",
        description=(
            "One or more Allow statements have no Condition block. "
            "Adding conditions (IP restrictions, MFA, VPC source) significantly reduces attack surface."
        ),
        affected=affected,
        passed=len(affected) == 0,
        remediation=(
            "Add conditions such as aws:SourceIp, aws:MultiFactorAuthPresent, "
            "or aws:SourceVpc to limit where and how access is granted."
        ),
        reference="https://docs.aws.amazon.com/IAM/latest/UserGuide/reference_policies_elements_condition.html",
    )


def check_old_policy_version(config: dict) -> Finding:
    """Checks if policy uses latest language version."""
    version = config.get("Version", "")
    passed = version == "2012-10-17"
    return Finding(
        id="IAM-005",
        title="Outdated Policy Language Version",
        severity="LOW",
        description=(
            f"Policy Version is '{version}'. The current recommended version is '2012-10-17', "
            "which enables policy variables and modern features."
        ),
        affected=[] if passed else [f"Version: '{version}'"],
        passed=passed,
        remediation='Set "Version": "2012-10-17" at the top of your policy document.',
        reference="https://docs.aws.amazon.com/IAM/latest/UserGuide/reference_policies_elements_version.html",
    )


# ─────────────────────────────────────────────
# RUNNER
# ─────────────────────────────────────────────

def run_all_checks(config: dict) -> list[Finding]:
    check_fns = [
        check_wildcard_action,
        check_wildcard_resource,
        check_public_principal,
        check_missing_mfa_condition,
        check_allow_s3_write_public,
        check_missing_condition_block,
        check_old_policy_version,
    ]
    findings = [fn(config) for fn in check_fns]
    findings.sort(key=lambda f: SEVERITY_ORDER.get(f.severity, 99))
    return findings

print("✅ checks.py loaded — 7 security checks ready")

## Step 3 — Load Report Generator

In [ ]:
"""
report.py — Generates HTML report sections from findings.
Returns three HTML blocks: summary, details, remediation.
"""

from app.checks import Finding

SEVERITY_COLORS = {
    "CRITICAL": "#ef4444",
    "HIGH":     "#f97316",
    "MEDIUM":   "#eab308",
    "LOW":      "#22c55e",
    "INFO":     "#3b82f6",
}

SEVERITY_BG = {
    "CRITICAL": "#2d1515",
    "HIGH":     "#2d1a0e",
    "MEDIUM":   "#2a2200",
    "LOW":      "#0f2a14",
    "INFO":     "#0d1f3c",
}

SEVERITY_ICONS = {
    "CRITICAL": "🔴",
    "HIGH":     "🟠",
    "MEDIUM":   "🟡",
    "LOW":      "🟢",
    "INFO":     "🔵",
}


def _badge(severity: str) -> str:
    color = SEVERITY_COLORS.get(severity, "#64748b")
    icon = SEVERITY_ICONS.get(severity, "⚪")
    return (
        f'<span style="background:{color}22; color:{color}; border:1px solid {color}66; '
        f'padding:2px 10px; border-radius:999px; font-size:0.72rem; '
        f'font-family:\'JetBrains Mono\',monospace; font-weight:600; letter-spacing:0.05em;">'
        f'{icon} {severity}</span>'
    )


def generate_report(findings: list[Finding]):
    failed = [f for f in findings if not f.passed]
    passed = [f for f in findings if f.passed]

    counts = {}
    for f in failed:
        counts[f.severity] = counts.get(f.severity, 0) + 1

    # ── RISK SCORE ──
    weights = {"CRITICAL": 40, "HIGH": 20, "MEDIUM": 10, "LOW": 5}
    raw_score = sum(weights.get(sev, 0) * cnt for sev, cnt in counts.items())
    risk_score = min(100, raw_score)
    risk_label = (
        "CRITICAL RISK" if risk_score >= 60
        else "HIGH RISK" if risk_score >= 40
        else "MEDIUM RISK" if risk_score >= 20
        else "LOW RISK" if risk_score > 0
        else "SECURE"
    )
    score_color = (
        "#ef4444" if risk_score >= 60
        else "#f97316" if risk_score >= 40
        else "#eab308" if risk_score >= 20
        else "#22c55e"
    )

    # ────────────────────────────────
    # SUMMARY TAB
    # ────────────────────────────────
    sev_rows = ""
    for sev in ["CRITICAL", "HIGH", "MEDIUM", "LOW"]:
        cnt = counts.get(sev, 0)
        color = SEVERITY_COLORS[sev]
        sev_rows += f"""
        <tr>
            <td style="padding:8px 12px;">{_badge(sev)}</td>
            <td style="padding:8px 12px; text-align:center; font-family:'JetBrains Mono',monospace;
                       font-size:1.1rem; font-weight:700; color:{color if cnt > 0 else '#64748b'};">
                {cnt}
            </td>
        </tr>"""

    failed_list = ""
    for f in failed:
        failed_list += f"""
        <li style="margin:6px 0; font-size:0.88rem;">
            {_badge(f.severity)} &nbsp;
            <span style="color:#e2e8f0;">{f.id}</span>
            <span style="color:#94a3b8;"> — {f.title}</span>
        </li>"""

    passed_list = ""
    for f in passed:
        passed_list += f'<li style="margin:4px 0; font-size:0.85rem; color:#22c55e;">✅ {f.id} — {f.title}</li>'

    summary_html = f"""
    <div style="font-family:'Syne',sans-serif; color:#e2e8f0; padding:1rem 0;">

        <!-- Risk Score Card -->
        <div style="background:#111827; border:1px solid {score_color}44; border-radius:12px;
                    padding:1.5rem 2rem; margin-bottom:1.5rem; display:flex;
                    align-items:center; gap:2rem;">
            <div style="text-align:center;">
                <div style="font-size:3rem; font-weight:800; color:{score_color};
                            font-family:'JetBrains Mono',monospace; line-height:1;">
                    {risk_score}
                </div>
                <div style="font-size:0.7rem; color:#64748b; letter-spacing:0.15em; margin-top:4px;">
                    RISK SCORE
                </div>
            </div>
            <div>
                <div style="font-size:1.3rem; font-weight:700; color:{score_color};">{risk_label}</div>
                <div style="color:#94a3b8; font-size:0.88rem; margin-top:4px;">
                    {len(failed)} issue{'s' if len(failed) != 1 else ''} detected &nbsp;·&nbsp;
                    {len(passed)} check{'s' if len(passed) != 1 else ''} passed
                </div>
            </div>
        </div>

        <!-- Severity Breakdown -->
        <div style="background:#111827; border:1px solid #1e2d45; border-radius:12px;
                    padding:1rem 1.5rem; margin-bottom:1.5rem;">
            <div style="font-family:'JetBrains Mono',monospace; color:#00d4ff;
                        font-size:0.7rem; letter-spacing:0.15em; margin-bottom:0.75rem;">
                SEVERITY BREAKDOWN
            </div>
            <table style="width:100%; border-collapse:collapse;">
                {sev_rows}
            </table>
        </div>

        <!-- Failed Checks -->
        {"" if not failed else f'''
        <div style="background:#111827; border:1px solid #1e2d45; border-radius:12px;
                    padding:1rem 1.5rem; margin-bottom:1.5rem;">
            <div style="font-family:\'JetBrains Mono\',monospace; color:#ff6b35;
                        font-size:0.7rem; letter-spacing:0.15em; margin-bottom:0.75rem;">
                FAILED CHECKS
            </div>
            <ul style="margin:0; padding-left:0.5rem; list-style:none;">{failed_list}</ul>
        </div>
        '''}

        <!-- Passed Checks -->
        {"" if not passed else f'''
        <div style="background:#111827; border:1px solid #1e2d45; border-radius:12px;
                    padding:1rem 1.5rem;">
            <div style="font-family:\'JetBrains Mono\',monospace; color:#22c55e;
                        font-size:0.7rem; letter-spacing:0.15em; margin-bottom:0.75rem;">
                PASSED CHECKS
            </div>
            <ul style="margin:0; padding-left:0.5rem; list-style:none;">{passed_list}</ul>
        </div>
        '''}
    </div>
    """

    # ────────────────────────────────
    # DETAILS TAB
    # ────────────────────────────────
    detail_cards = ""
    for f in findings:
        status_icon = "✅" if f.passed else "❌"
        bg = "#0f2a14" if f.passed else SEVERITY_BG.get(f.severity, "#1a1a2e")
        border_color = "#22c55e44" if f.passed else SEVERITY_COLORS.get(f.severity, "#64748b") + "44"

        affected_html = ""
        if f.affected:
            items = "".join(
                f'<li style="font-family:\'JetBrains Mono\',monospace; font-size:0.78rem; '
                f'color:#94a3b8; margin:3px 0; background:#0a0e1a; padding:4px 8px; '
                f'border-radius:4px; word-break:break-all;">{item}</li>'
                for item in f.affected
            )
            affected_html = f"""
            <div style="margin-top:0.75rem;">
                <div style="font-size:0.72rem; color:#64748b; letter-spacing:0.1em;
                            margin-bottom:4px; font-family:'JetBrains Mono',monospace;">
                    AFFECTED
                </div>
                <ul style="margin:0; padding-left:0.5rem; list-style:none;">{items}</ul>
            </div>"""

        detail_cards += f"""
        <div style="background:{bg}; border:1px solid {border_color}; border-radius:10px;
                    padding:1rem 1.25rem; margin-bottom:1rem;">
            <div style="display:flex; align-items:center; gap:10px; margin-bottom:0.5rem;">
                <span style="font-family:'JetBrains Mono',monospace; color:#64748b;
                             font-size:0.75rem;">{f.id}</span>
                {_badge(f.severity)}
                <span style="margin-left:auto; font-size:1rem;">{status_icon}</span>
            </div>
            <div style="font-weight:700; font-size:0.98rem; color:#e2e8f0; margin-bottom:0.5rem;">
                {f.title}
            </div>
            <div style="color:#94a3b8; font-size:0.87rem; line-height:1.6;">
                {f.description}
            </div>
            {affected_html}
        </div>"""

    details_html = f"""
    <div style="font-family:'Syne',sans-serif; color:#e2e8f0; padding:1rem 0;">
        {detail_cards if detail_cards else '<p style="color:#64748b;">No findings to display.</p>'}
    </div>"""

    # ────────────────────────────────
    # REMEDIATION TAB
    # ────────────────────────────────
    rem_cards = ""
    failed_findings = [f for f in findings if not f.passed]
    if not failed_findings:
        rem_cards = """
        <div style="text-align:center; padding:3rem; color:#22c55e;">
            <div style="font-size:3rem; margin-bottom:1rem;">✅</div>
            <div style="font-size:1.1rem; font-weight:700;">No remediations needed.</div>
            <div style="color:#64748b; margin-top:0.5rem;">All checks passed.</div>
        </div>"""
    else:
        for i, f in enumerate(failed_findings, 1):
            color = SEVERITY_COLORS.get(f.severity, "#64748b")
            ref_link = (
                f'<a href="{f.reference}" target="_blank" '
                f'style="color:#00d4ff; font-size:0.8rem; font-family:\'JetBrains Mono\',monospace;">'
                f'AWS Docs →</a>'
            ) if f.reference else ""

            rem_cards += f"""
            <div style="background:#111827; border-left:3px solid {color};
                        border-radius:0 10px 10px 0; padding:1rem 1.25rem; margin-bottom:1rem;">
                <div style="display:flex; align-items:center; gap:8px; margin-bottom:0.5rem;">
                    <span style="font-family:'JetBrains Mono',monospace; color:#64748b;
                                 font-size:0.72rem;">#{i} · {f.id}</span>
                    {_badge(f.severity)}
                </div>
                <div style="font-weight:700; color:#e2e8f0; margin-bottom:0.5rem;">{f.title}</div>
                <div style="color:#94a3b8; font-size:0.87rem; line-height:1.6;
                            background:#0a0e1a; padding:10px 14px; border-radius:6px;
                            font-family:'JetBrains Mono',monospace; white-space:pre-wrap;">
                    {f.remediation}
                </div>
                {"" if not ref_link else f'<div style="margin-top:0.6rem;">{ref_link}</div>'}
            </div>"""

    remediation_html = f"""
    <div style="font-family:'Syne',sans-serif; color:#e2e8f0; padding:1rem 0;">
        {rem_cards}
    </div>"""

    return summary_html, details_html, remediation_html

print("✅ report.py loaded")

## Step 4 — Load Showcase Test Configs

In [ ]:
import json

test_full_disaster = {
    "Version": "2008-10-17",
    "Statement": [{"Effect": "Allow", "Action": "*", "Resource": "*", "Principal": "*"}]
}

test_partial_risk = {
    "Version": "2012-10-17",
    "Statement": [
        {"Sid": "DevTeamAccess", "Effect": "Allow",
         "Action": ["iam:CreateUser", "iam:DeleteUser", "ec2:*"],
         "Resource": "*",
         "Principal": {"AWS": "arn:aws:iam::123456789012:user/dev-lead"}},
        {"Sid": "PublicReads", "Effect": "Allow",
         "Action": ["s3:GetObject"],
         "Resource": "arn:aws:s3:::company-reports/*",
         "Principal": "*"}
    ]
}

test_secure = {
    "Version": "2012-10-17",
    "Statement": [{
        "Sid": "ScopedReadOnly", "Effect": "Allow",
        "Action": ["s3:GetObject", "s3:ListBucket"],
        "Resource": ["arn:aws:s3:::my-secure-bucket", "arn:aws:s3:::my-secure-bucket/reports/*"],
        "Principal": {"AWS": "arn:aws:iam::123456789012:role/ReportReaderRole"},
        "Condition": {
            "Bool": {"aws:MultiFactorAuthPresent": "true"},
            "StringEquals": {"aws:SourceVpc": "vpc-0a1b2c3d4e5f"}
        }
    }]
}

print("✅ 3 showcase test configs loaded")
print("   → test_full_disaster  : expects Risk Score 100 · CRITICAL RISK")
print("   → test_partial_risk   : expects Risk Score ~60 · HIGH RISK")
print("   → test_secure         : expects Risk Score 0   · SECURE")


## 🔴 Showcase Test 1 — Full Disaster
**Config:** Wildcard action + resource + public principal + outdated version + no conditions.  
**Expected:** Risk Score 100 · CRITICAL RISK · all 7 checks triggered.

In [ ]:
findings1 = run_all_checks(test_full_disaster)
failed1 = [f for f in findings1 if not f.passed]
passed1 = [f for f in findings1 if f.passed]
print("TEST 1 — FULL DISASTER")
print("─" * 45)
print(f"Failed: {len(failed1)}  |  Passed: {len(passed1)}")
print()
for f in findings1:
    status = "❌ FAIL" if not f.passed else "✅ PASS"
    print(f"  {status}  [{f.severity:<8}]  {f.id}  —  {f.title}")


## 🟠 Showcase Test 2 — Partial Risk
**Config:** Named principal but broad IAM/EC2 actions + public S3 read + no MFA, no conditions.  
**Expected:** Risk Score ~60 · HIGH RISK · 4 findings.

In [ ]:
findings2 = run_all_checks(test_partial_risk)
failed2 = [f for f in findings2 if not f.passed]
passed2 = [f for f in findings2 if f.passed]
print("TEST 2 — PARTIAL RISK")
print("─" * 45)
print(f"Failed: {len(failed2)}  |  Passed: {len(passed2)}")
print()
for f in findings2:
    status = "❌ FAIL" if not f.passed else "✅ PASS"
    print(f"  {status}  [{f.severity:<8}]  {f.id}  —  {f.title}")


## 🟢 Showcase Test 3 — Secure Policy
**Config:** Scoped actions, specific ARN, named role, MFA + VPC conditions.  
**Expected:** Risk Score 0 · SECURE · all 7 checks pass.

In [ ]:
findings3 = run_all_checks(test_secure)
failed3 = [f for f in findings3 if not f.passed]
passed3 = [f for f in findings3 if f.passed]
print("TEST 3 — SECURE POLICY")
print("─" * 45)
print(f"Failed: {len(failed3)}  |  Passed: {len(passed3)}")
print()
for f in findings3:
    status = "❌ FAIL" if not f.passed else "✅ PASS"
    print(f"  {status}  [{f.severity:<8}]  {f.id}  —  {f.title}")


## Step 5 — Launch the Interactive App (Live Share)
Run this cell to start Gradio with a **public shareable link** valid for 72 hours.  
All 3 showcase configs are pre-loaded in the sample dropdown.

In [ ]:
import gradio as gr
import json

def analyze_config(file_obj, raw_text):
    content = None
    if file_obj is not None:
        with open(file_obj.name, "r") as f:
            content = f.read()
    elif raw_text and raw_text.strip():
        content = raw_text.strip()
    else:
        return "<p style='color:orange;'>⚠️ Please upload a file or paste config content.</p>","",""
    try:
        config = json.loads(content)
    except json.JSONDecodeError as e:
        return f"<p style='color:red;'>❌ Invalid JSON: {e}</p>","",""
    findings = run_all_checks(config)
    return generate_report(findings)

def load_sample(name):
    samples = {
        "🔴 Full Disaster": json.dumps(test_full_disaster, indent=2),
        "🟠 Partial Risk":  json.dumps(test_partial_risk, indent=2),
        "🟢 Secure Policy": json.dumps(test_secure, indent=2),
    }
    return samples.get(name, "")

with gr.Blocks(title="Cloud Security Posture Checker") as demo:
    gr.HTML("""
    <div style='text-align:center;padding:1.5rem 0 1rem;background:#0a0e1a;border-radius:12px;margin-bottom:1rem;'>
      <div style='color:#00d4ff;font-size:0.75rem;letter-spacing:0.2em;margin-bottom:0.5rem;'>CLOUD SECURITY POSTURE CHECKER</div>
      <h1 style='font-size:2rem;font-weight:800;color:#e2e8f0;margin:0;'>AWS Config <span style='color:#00d4ff;'>Analyzer</span></h1>
      <p style='color:#64748b;margin-top:0.5rem;font-size:0.9rem;'>Detect misconfigurations in IAM & S3 policies · Built by Kobia Williams</p>
    </div>""")
    with gr.Row():
        with gr.Column(scale=1):
            file_input  = gr.File(label="📁 Upload JSON Policy File", file_types=[".json"])
            gr.HTML('<div style="text-align:center;color:#64748b;margin:0.5rem 0;">— or paste below —</div>')
            text_input  = gr.Code(label="Paste JSON Content", language="json", lines=12)
            with gr.Row():
                sample_dd = gr.Dropdown(choices=["🔴 Full Disaster","🟠 Partial Risk","🟢 Secure Policy"], label="Load Showcase Sample")
                load_btn  = gr.Button("Load", size="sm")
            analyze_btn = gr.Button("🔍 Analyze Config", variant="primary", size="lg")
            clear_btn   = gr.Button("Clear", size="sm")
        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.Tab("📊 Summary"):     summary_out     = gr.HTML()
                with gr.Tab("🔎 Findings"):    details_out     = gr.HTML()
                with gr.Tab("🛠 Remediation"): remediation_out = gr.HTML()
    analyze_btn.click(analyze_config, [file_input, text_input], [summary_out, details_out, remediation_out])
    load_btn.click(load_sample, [sample_dd], [text_input])
    clear_btn.click(lambda: (None,"","","",""), [], [file_input, text_input, summary_out, details_out, remediation_out])
    gr.HTML('<div style="text-align:center;padding:1rem 0;color:#64748b;font-size:0.8rem;">Built by <b style="color:#00d4ff;">Kobia Williams</b> · Cloud Security Portfolio · <a href="https://github.com/KobiaW/Cloud-Security-Posture-Checker" style="color:#00d4ff;">GitHub ↗</a></div>')

demo.launch(share=True)
